# **🏠 Heritage Housing — House Price Data Study**

---

## 📋 1. House Price Data Study Summary

### Objectives

- The objective of this notebook is to address Business Requirement 1 by investigating which property features have the strongest relationship with house sale prices.
- The analysis will use correlation analysis and Predictive Power Score (PPS) to identify the most relevant variables.
- The strongest relationships will then be visualised against SalePrice.

### Inputs

- outputs/datasets/collection/house_prices_records.csv
- outputs/datasets/collection/inherited_houses.csv

### Outputs

- generate code that answers business requirement 1 and can be used to build the Streamlit App

---

## ⚙️ 2. Set Up the Notebook

## Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/milestone-project5-heritage-housing-issues/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/workspaces/milestone-project5-heritage-housing-issues'

---

## 📥 3. Load Data

Import heritage housing historical data.

In [4]:
import pandas as pd

df = pd.read_csv("outputs/datasets/collection/house_prices_records.csv")

df.head(3)

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500


---

## 🔍 4. Data Exploration

In [15]:
from ydata_profiling import ProfileReport
pandas_report = ProfileReport(df=df, minimal=True)
pandas_report.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Interpretation of Pandas Profiling Report
* **24 variables** in total
* **20 numerical variables**
* **4 categorical variables**
  *  `BsmtExposure` – Basement exposure
  *  `BsmtFinType1` – Type of basement finish
  *  `GarageFinish` – Garage finish level
  *  `KitchenQual` – Kitchen quality
* `OverallQual` and `OverallCond` are numerical variables with an **ordinal interpretation**.
* `BedroomAbvGr` is also a numerical variable, but it represents a **count** rather than a continuous measurement.
* There are **missing values in several variables**, particularly `EnclosedPorch`, `WoodDeckSF`, `LotFrontage`, `GarageFinish`, and `BsmtFinType1`.

### Missing Values
- First, I checked the number of missing values in each feature:

In [8]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing

EnclosedPorch    1324
WoodDeckSF       1305
LotFrontage       259
GarageFinish      235
BsmtFinType1      145
BedroomAbvGr       99
2ndFlrSF           86
GarageYrBlt        81
BsmtExposure       38
MasVnrArea          8
dtype: int64

- Not all `NaN` values necessarily represent unknown data. For some categorical variables, `NaN` can indicate that the feature **does not apply** to the property.
- I investigated the missing values in the garage and basement-related categorical variables:

In [9]:
for col in ['GarageFinish', 'BsmtFinType1', 'BsmtExposure']:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


GarageFinish
GarageFinish
Unf    546
RFn    366
Fin    313
NaN    235
Name: count, dtype: int64

BsmtFinType1
BsmtFinType1
Unf    396
GLQ    385
ALQ    202
NaN    145
BLQ    136
Rec    126
LwQ     70
Name: count, dtype: int64

BsmtExposure
BsmtExposure
No     953
Av     221
Gd     134
Mn     114
NaN     38
Name: count, dtype: int64


These missing values are likely to represent **absence of the corresponding feature**:

- `GarageFinish`: the property likely has no garage.
- `BsmtFinType1`: the property likely has no basement.
- `BsmtExposure`: the property likely has no basement.

I therefore replaced these `NaN` values with an explicit `"None"` category:

In [11]:
df['GarageFinish'] = df['GarageFinish'].fillna('None')
df['BsmtFinType1'] = df['BsmtFinType1'].fillna('None')
df['BsmtExposure'] = df['BsmtExposure'].fillna('None')

## Handle Missing Numerical Values

Before correlation and PPS analysis, the remaining missing numerical values are handled based on the meaning of each feature.

### Imputation Strategy

* **Replace with `0`:** `EnclosedPorch`, `WoodDeckSF`, `2ndFlrSF`, and `MasVnrArea`, where missing values can reasonably represent the absence of the feature.
* **Replace with the median:** `LotFrontage`, `BedroomAbvGr`, and `GarageYrBlt`, where `0` would not be a realistic replacement.

The median is used for genuinely missing numerical values as it is less sensitive to outliers than the mean.

After imputation, the dataset will be checked to confirm that no missing values remain.


In [13]:
# Fill missing values where NaN represents the absence of a feature
absence_cols = [
    'EnclosedPorch',
    'WoodDeckSF',
    '2ndFlrSF',
    'MasVnrArea'
]

df[absence_cols] = df[absence_cols].fillna(0)


# Fill genuinely missing numerical values with the median
median_cols = [
    'LotFrontage',
    'BedroomAbvGr',
    'GarageYrBlt'
]

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

In [14]:
df.isnull().sum().sort_values(ascending=False)

1stFlrSF         0
2ndFlrSF         0
BedroomAbvGr     0
BsmtExposure     0
BsmtFinSF1       0
BsmtFinType1     0
BsmtUnfSF        0
EnclosedPorch    0
GarageArea       0
GarageFinish     0
GarageYrBlt      0
GrLivArea        0
KitchenQual      0
LotArea          0
LotFrontage      0
MasVnrArea       0
OpenPorchSF      0
OverallCond      0
OverallQual      0
TotalBsmtSF      0
WoodDeckSF       0
YearBuilt        0
YearRemodAdd     0
SalePrice        0
dtype: int64

---

## 🧹 5. 

---

## 💾 6. 

---

## ✅ 7. Conclusions